# Standalone Concept Mel Visualization

This notebook visualizes concept `.npy` files directly.

- No dependency on project visualization `.py` files
- Reads concept folders from disk
- Shows random examples per concept

In [ ]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Settings ---
BASE_DIR = Path("concepts")
RANDOM_SEED = 1337
SAMPLES_PER_CONCEPT = 6   # how many random samples to show per concept
FIG_SCALE = 2.6

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if not BASE_DIR.exists():
    raise FileNotFoundError(f"Concept base dir not found: {BASE_DIR.resolve()}")

concept_dirs = sorted([d for d in BASE_DIR.iterdir() if d.is_dir()])
print(f"Found {len(concept_dirs)} concept folders in: {BASE_DIR.resolve()}")
for d in concept_dirs:
    n = len(list(d.glob("*.npy")))
    print(f"  - {d.name}: {n} npy files")

In [ ]:
def pick_samples(concept_dir: Path, k: int) -> list[Path]:
    files = sorted([p for p in concept_dir.glob("*.npy") if p.name != "_preview_stack.npy"])
    if not files:
        return []
    if len(files) <= k:
        return files
    return random.sample(files, k)


def show_grid(arrays: list[np.ndarray], title: str) -> None:
    if not arrays:
        print(f"No arrays to show for: {title}")
        return
    n = len(arrays)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_SCALE * ncols, FIG_SCALE * nrows), squeeze=False)
    for i, arr in enumerate(arrays):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        im = ax.imshow(arr, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=1)
        ax.set_title(f"sample {i+1}", fontsize=9)
        ax.set_xlabel("Time frame")
        ax.set_ylabel("Mel bin")

    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show random samples for each concept
for concept_dir in concept_dirs:
    sample_paths = pick_samples(concept_dir, SAMPLES_PER_CONCEPT)
    arrays = [np.load(p) for p in sample_paths]
    print(f"\nConcept: {concept_dir.name} | showing {len(arrays)} samples")
    show_grid(arrays, title=f"{concept_dir.name}")

In [ ]:
# Optional: one summary panel (mean map per concept)
means = []
names = []
for concept_dir in concept_dirs:
    files = sorted([p for p in concept_dir.glob("*.npy") if p.name != "_preview_stack.npy"])
    if not files:
        continue
    stack = np.stack([np.load(p) for p in files], axis=0)
    means.append(stack.mean(axis=0))
    names.append(concept_dir.name)

if means:
    n = len(means)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.2 * nrows), squeeze=False)
    for i, (m, name) in enumerate(zip(means, names)):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        im = ax.imshow(m, aspect="auto", origin="lower", cmap="jet", vmin=0, vmax=1)
        ax.set_title(name, fontsize=9)
        ax.set_xlabel("Time")
        ax.set_ylabel("Mel")
    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")
    fig.suptitle("Mean Mel Map per Concept", fontsize=12, fontweight="bold")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.show()
else:
    print("No concept files found to compute means.")